# Exercise 2.2.3 — Cleaning, Missing Values & Duplicates

This notebook continues from the **typed checkpoint** produced by 2.2.2 (`data/10_cleaned/datania_households_clean.csv`). Types are already fixed — income is numeric, dates are parsed, category typos are corrected. Now you handle the messiness that *types alone don't fix*: missing values, coded/sentinel values, duplicates, and impossible values.

You will practice:
- Working on a **copy** and documenting every change
- Detecting missing values with `isna().sum()`, percentages, and row inspection
- Recoding **coded / sentinel missing values** (`99`, `999`, `9999`, `999999`) and recovering sign-entry errors with `.abs()`
- Choosing a missing-value strategy: drop, drop a subset, or fill
- Detecting and removing **exact** and **subset** duplicates
- Applying **validation rules** to catch impossible values
- Saving the fully cleaned dataset back to `10_cleaned/`

> **Pipeline:** reads the typed checkpoint from `10_cleaned/` and overwrites it with the cleaned dataset. Exercise 2.2.4 reads from there.

### Path Setup (run first)

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
clean_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

# Typed checkpoint from 2.2.2 — income is numeric, dates parsed, category typos fixed
df = pd.read_csv(clean_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded typed checkpoint:', df.shape)
df.head()

---

## Task 1 — Never destroy the input: make a clean copy

Cleaning operations should never touch the DataFrame you loaded. Create a separate working copy and apply every change to it. This keeps the checkpoint intact for comparison and lets you restart if your assumptions change.

In [ ]:
# Create the working copy
df_clean = df.  # your code here

print('Working copy:', df_clean.shape)

**The golden rule of notebook documentation**

- **Above each code cell** (Markdown): explain *what* you are about to do and *why*.
- **Below each code cell** (Markdown): interpret the *result* and note the decision you made.

---

## Task 2 — Detect missing values

`isna().sum()` counts blanks/`NaN` per column. The percentage version shows how serious each gap is, and inspecting the affected rows shows *which* records are incomplete. Because 2.2.2 already converted `income_dkw` and `survey_date`, their text placeholders (`unknown`, `not recorded`) are already real `NaN`/`NaT` here.

In [ ]:
# Count missing values per column
df_clean.  # your code here

In [ ]:
# Percentage missing per column (round to 2 decimals)
(df_clean.isna().sum() / # your code here ).round(2)

In [ ]:
# Show every row that has at least one missing value
df_clean[df_clean.isna(). # your code here ]

**Questions:**

- Which columns have missing values? How many rows are affected?
- Which gaps look serious enough to worry about, and which are negligible?

---

## Task 3 — Coded missing values

Survey data mixes a few problems that all look like numbers:

| Value | Meaning | Fix |
|---|---|---|
| `-5000` | negative income — a **sign-entry error** | recover with `.abs()` |
| `999999` | all-nines "not stated" **sentinel** | recode to `NaN` |
| `999`, `9999`, `99` | column sentinels (age, density, education) | recode to `NaN` |

Not every odd value is missing: a negative income is a fixable typo whose magnitude is real, while a sentinel carries no real value. Handle each appropriately and check the impact.

In [ ]:
# A negative income is a sign-entry error — recover the magnitude (it is NOT missing)
df_clean['income_dkw'] = df_clean['income_dkw'].  # your code here — .abs()

# 999999 is an all-nines "not stated" sentinel — recode it to NaN
df_clean['income_dkw'] = df_clean['income_dkw'].replace( # your code here — 999999, np.nan )

df_clean['income_dkw'].describe()

In [ ]:
# A reusable helper keeps the recoding consistent across columns
def recode_coded_missing(series, codes):
    """Replace coded missing values with NaN."""
    return series.replace(codes, np.nan)

print('Mean age BEFORE recode:', round(df_clean['age'].mean(), 1))

df_clean['age'] = recode_coded_missing(df_clean['age'], [999])
df_clean['pop_density'] = recode_coded_missing(df_clean['pop_density'], # your code here — [9999] )
df_clean['education_code'] = recode_coded_missing(df_clean['education_code'], # your code here — [99] )

print('Mean age AFTER recode: ', round(df_clean['age'].mean(), 1))

**Questions:**

- Why recover the negative income with `.abs()` but recode `999999` to `NaN`? What makes one a fixable error and the other truly missing?
- By how much did the mean age change after recoding `999`? What does that say about leaving sentinels in place?
- Why is `0` a *tricky* code (think `hh_size` or `income`)?

---

## Task 4 — Missing-value strategy: drop critical-only

Three strategies exist: drop any row with a gap, drop only rows missing a **critical** column, or **fill** the gaps. Dropping everything is usually too aggressive. Here the only non-negotiable column is the identifier `hh_id` — drop rows missing it, leave the rest.

In [ ]:
print('Before:', df_clean.shape)
df_clean = df_clean.dropna(subset= # your code here )
print('After: ', df_clean.shape)

In [ ]:
# A fill is sometimes appropriate. Preview only (do NOT overwrite df_clean):
example = df_clean['income_dkw']. # your code here — fillna with the median
print('NaN before fill:', df_clean['income_dkw'].isna().sum(),
      '| NaN after fill:', example.isna().sum())

**Questions:**

- Did `dropna(subset=['hh_id'])` remove any rows here? Why is it still worth running?
- Filling income with the median changes the distribution. When is that acceptable, and when does it introduce bias?

---

## Task 5 — Detect and remove duplicates

Real surveys record the same household twice. First handle **exact** duplicates (every column identical), then **subset** duplicates (same `hh_id`, different other fields).

In [ ]:
# How many fully-identical rows are there? Show them.
print('Exact duplicate rows:', df_clean.duplicated().sum())
df_clean[df_clean.duplicated( # your code here — keep=False )].sort_values('hh_id')

In [ ]:
print('Before:', df_clean.shape)
df_clean = df_clean. # your code here — drop_duplicates()
print('After: ', df_clean.shape)

In [ ]:
# Subset duplicates: same hh_id, but the rows differ. Show all occurrences.
dups = df_clean[df_clean.duplicated(subset= # your code here , keep=False)]
dups.sort_values('hh_id')[['hh_id', 'district', 'income_dkw', 'survey_date']]

When two rows share an `hh_id` but differ, choose which to keep. A robust rule is **keep the most complete** record (the one with the fewest missing values).

In [ ]:
df_clean['missing_count'] = df_clean.isna().sum(axis=1)
df_clean = (
    df_clean
    .sort_values(['hh_id', 'missing_count'])
    .drop_duplicates(subset=['hh_id'], keep= # your code here — 'first' )
)
df_clean = df_clean.drop(columns='missing_count')
print('After resolving subset duplicates:', df_clean.shape)

**Questions:**

- Which `hh_id` was an exact duplicate? Which was a subset duplicate?
- For the subset duplicate, which record was kept and why? What other rule could you use (hint: `survey_date`)?
- Why does `keep=False` matter when you are *inspecting* duplicates?

---

## Task 6 — Validation rules: catch impossible values

Some values are not missing — they are *impossible*. A household cannot have `0`, `-1`, or `99` members. Filter out the rows that fail a plausibility rule, checking the shape before and after.

In [ ]:
print('hh_size values:', sorted(df_clean['hh_size'].unique()))
print('Before:', df_clean.shape)
df_clean = df_clean[df_clean['hh_size'].between( # your code here — 1, 20 )]
print('After: ', df_clean.shape)

**Questions:**

- How many rows did the `hh_size` rule remove? Which households were they?
- `age` was already handled by sentinel recoding in Task 3. What is the difference between *recoding to NaN* and *dropping the row*?

---

## Task 7 — Save the cleaned dataset

Overwrite the checkpoint in `10_cleaned/` with the fully cleaned result, then reload it to confirm it round-trips. (Raw data in `0_raw/` is never touched.)

In [ ]:
print('Final cleaned shape:', df_clean.shape)
df_clean.isna().sum()

In [ ]:
df_clean = df_clean.reset_index(drop=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_clean.to_csv( # your code here — index=False )
print('Saved:', out_path)

In [ ]:
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded shape:', check.shape)
print()
print(check.dtypes)
check.head()

**Questions:**

- How many rows survived the full cleaning pipeline, starting from the 28-row checkpoint?
- After reloading, what dtype does `survey_date` have? What would you do before using it for date analysis?
- Could you justify every removed row to a reviewer?